# Projet 3 Les volcans remplissent un réservoir

**Question.** Combien de CO2 peut entrer dans l’atmosphère sans inventer du carbone ? **Ancrage :** document 1, § 9–13 ; document 5, réservoirs. Suite et conservation.

**Entrées.** state : mantle_c_mol, co2_mol ; params : volcanic_mol_yr. dt_yr doit être strictement positif. Les sorties N2 et vapeur du dégazage réel restent pour un prolongement avec réserves explicites.

**Modèle minimal.** Le flux F demandé est constant. Le flux réellement transféré est min(F, mantle_c_mol/dt_yr). Une mole de carbone du manteau devient une mole de CO2 atmosphérique. Notre réservoir est une réserve mobilisable fictive, pas tout le carbone réel du manteau.

**Sorties.** rates : co2_mol = +F_effectif, mantle_c_mol = −F_effectif. diag peut rester vide. Le moteur applique les deux taux ensemble.

**Données d’essai.** Stock du manteau : 100 mol ; CO2 : 0 mol ; F = 10 mol/an ; dt = 1 an. Utiliser de petites valeurs pour voir le bilan, puis changer d’échelle. Pour une expérience géologique fictive : 1e20 mol mobilisables et F = 1e13 mol/an.

**Travail.** Simuler 15 ans ; produire la courbe des deux réservoirs ; augmenter dt ; supprimer le flux. Ne pas appeler l’horloge depuis la routine.

**Validation.** Après 5 ans dans l’essai, les stocks valent 50 et 50 mol. Après 10 ans, le manteau est épuisé et le CO2 vaut 100 mol. Leur somme reste 100. Si dt = 20 ans, le transfert maximal reste 100 mol.

**Rendu.** Routine degazage.py, courbes et bilan de conservation. **Prolongement ⊕⊕ :** argon issu du potassium avec la loi de demi-vie du cours et la branche de 10,7 % ; créer des stocks distincts plutôt que d’ajouter l’argon au carbone.

Lire aussi le contrat commun et le complément eau du dossier collectif.


## Travailler dans le navigateur

Ouvrir ce fichier .ipynb dans https://notebook.basthon.fr ou Colab. Aucun fichier Python annexe ni installation locale ne sont requis. Exécuter les cellules de haut en bas avec Maj + Entrée. Compléter uniquement les zones **À VOUS**, puis relancer les cellules qui suivent. Télécharger le notebook à la fin et le déposer dans votre dossier OneDrive G03. Le notebook ne se sauvegarde pas automatiquement dans OneDrive.

Les fonctions de référence du banc d’essai sont fournies ; elles ne sont pas la routine de votre groupe. Les formules sont dans la fiche ci-dessus et le complément commun.

## À VOUS — raisonnement avant le code

Membres et rôles : …

Entrées, sorties et unités : …

Formule symbolique et hypothèses : …

Calcul manuel : données … ; résultat attendu …

Cas limite prévu : …

In [ ]:
from copy import deepcopy
from math import exp, pi, isfinite, isclose
from random import Random
import matplotlib.pyplot as plt
def base():
    s = {"time_yr": 0., "T_K": 290., "ice": 0.,
         "co2_mol": 7.49e16, "n2_mol": 1.39e20, "o2_mol": 3.74e19,
         "h2o_mol": 0., "mantle_c_mol": 1e20, "rock_c_mol": 0.,
         "fe_mol": 0., "oxide_fe_mol": 0., "population": 0.,
         "fossil_J": 0., "emitted_c_mol": 0., "diag": {}}
    p = {"g_m_s2": 9.81, "area_m2": 5.10e14, "G": 6.67e-11,
         "kB": 1.38e-23, "NA": 6.02e23, "h": 6.63e-34, "c": 3e8,
         "mass_kg": 5.97e24, "radius_m": 6.371e6, "T_exo_K": 1000.,
         "S_W_m2": 1361., "sigma": 5.67e-8, "pref_Pa": 42.,
         "tau_T_yr": 10., "tau_ice_yr": 20.,
         "water_mode": "mixed_screen", "ocean_prescribed": True,
         "volcanic_mol_yr": 0., "weather_mol_yr": 0.,
         "land_fraction": 1., "co2_ref_mol": 7.49e16,
         "oxygen_mol_yr": 0., "lambda_nm": 255.,
         "ozone_column_mol_m2": .10, "absorption_m2_mol": 100.,
         "need_J_person_yr": 1e9, "renewable_J_yr": 4e11,
         "fossil_capacity_J_yr": 3e11, "carbon_mol_J": 2e-6,
         "birth_rate_yr": 0., "death_rate_yr": 0., "deficit_death_rate_yr": 0.}
    return s, p
"""Un moteur deterministe : lire, additionner les flux, avancer, verifier."""
from copy import deepcopy
from math import isfinite

STOCKS = ["co2_mol", "n2_mol", "o2_mol", "h2o_mol", "mantle_c_mol",
          "rock_c_mol", "fe_mol", "oxide_fe_mol", "population", "fossil_J",
          "emitted_c_mol"]
VARIABLES = STOCKS + ["T_K", "ice"]

def verifier_etat(s):
    for cle in VARIABLES + ["time_yr"]:
        if not isfinite(s[cle]) or s[cle] < 0:
            raise ValueError(f"Valeur invalide : {cle} = {s[cle]}")
    if s["ice"] > 1 or s["T_K"] <= 0:
        raise ValueError("Fraction de glace ou temperature invalide")

def verifier_diagnostic(x):
    if isinstance(x, dict):
        for valeur in x.values():
            verifier_diagnostic(valeur)
    elif isinstance(x, (int, float)) and not isfinite(x):
        raise ValueError("Diagnostic non fini")
    elif not isinstance(x, (str, bool, int, float, type(None))):
        raise ValueError("Type de diagnostic invalide")

def evaluer(s, p, modules, dt):
    travail = deepcopy(s)
    travail["diag"] = {}
    taux = {cle: 0.0 for cle in VARIABLES}
    for module in modules:
        try:
            r = module(deepcopy(travail), deepcopy(p), dt)
            if set(r) != {"rates", "values", "diag"} or r["values"]:
                raise ValueError("Contrat de sortie incorrect")
            for cle, valeur in r["rates"].items():
                if cle not in taux or not isfinite(valeur):
                    raise ValueError(f"Taux invalide : {cle}")
                taux[cle] += valeur
            for cle, valeur in r["diag"].items():
                if cle in travail["diag"]:
                    raise ValueError(f"Diagnostic duplique : {cle}")
                verifier_diagnostic(valeur)
                travail["diag"][cle] = valeur
        except (ValueError, KeyError, TypeError) as erreur:
            raise ValueError(f"{module.__module__}, t={s['time_yr']}, dt={dt} : {erreur}") from erreur
    return travail, taux

def planet_evolution(initial, params, modules, steps, dt_yr):
    if not isfinite(dt_yr) or dt_yr <= 0:
        raise ValueError("Le pas doit etre positif et fini")
    if not isinstance(steps, int) or steps < 0:
        raise ValueError("Le nombre de pas doit etre entier et >= 0")
    s = deepcopy(initial)
    verifier_etat(s)
    histoire = []
    for numero in range(steps + 1):
        # On enregistre les diagnostics du MEME instant que les stocks.
        travail, taux = evaluer(s, params, modules, dt_yr)
        histoire.append(deepcopy(travail))
        if numero == steps:
            break
        suivant = deepcopy(travail)
        for cle in VARIABLES:
            suivant[cle] = s[cle] + dt_yr * taux[cle]
        suivant["time_yr"] += dt_yr
        verifier_etat(suivant)
        s = suivant
    return histoire

def simuler_isole(initial, params, routine, steps, dt):
    """Banc d'essai : diagnostics voisins prescrits et conservés.
    Ce banc ne remplace pas le moteur collectif, qui recalcule les voisins.
    """
    s=deepcopy(initial);h=[deepcopy(s)]
    for _ in range(steps):
        r=routine(deepcopy(s),deepcopy(params),dt)
        assert set(r)=={'rates','values','diag'} and not r['values']
        for cle,taux in r['rates'].items():
            s[cle]+=dt*taux
            assert isfinite(s[cle]) and s[cle]>=0,cle
        s['time_yr']+=dt
        if 'ice' in s:assert s['ice']<=1
        h.append(deepcopy(s))
    return h


## À VOUS — écrire la routine

Lire les entrées de la fiche, calculer la relation et traiter les cas limites. Retourner rates, values vide et diag avec les clés exactes du contrat.

Remplacer le `raise NotImplementedError` par votre code. Les tests indiquent les valeurs attendues ; ils ne donnent pas votre implémentation.

In [ ]:
def step(s, p, dt):
    # À VOUS : écrire ici le modèle donné dans la fiche.
    raise NotImplementedError("routine du groupe 3")


## Vérifier votre fonction

Avant de coder, la cellule affiche « À compléter ». Une fois la routine écrite, une assertion qui échoue signifie que le test doit être examiné. Noter **attendu / obtenu / conclusion**, même quand les assertions passent.

In [ ]:
try:
    s,p=base();s.update(mantle_c_mol=100.,co2_mol=0.);p['volcanic_mol_yr']=10.
    h=planet_evolution(s,p,[step],15,1);assert h[5]['co2_mol']==50
    assert h[-1]['co2_mol']==100 and h[-1]['mantle_c_mol']==0
    assert all(isclose(a['co2_mol']+a['mantle_c_mol'],100) for a in h)
    assert step(s,p,20)['rates']['co2_mol']==5
    print("Vérifications numériques réussies. Ajouter votre propre essai ci-dessous.")
    s,p=base(); s0,p0=deepcopy(s),deepcopy(p)

    s0,p0=deepcopy(s),deepcopy(p);step(s,p,1)
    assert s==s0 and p==p0,"La routine a modifié les entrées"
except NotImplementedError as erreur:
    print("À compléter :", erreur)


## À VOUS — ajouter une vérification justifiée

Quel paramètre changez-vous ? Quel résultat prévoyez-vous et pourquoi ?

In [ ]:
# À VOUS : ajouter un test avec un résultat attendu et une justification.


## Première figure fournie comme point de départ

La cellule propose une expérience et des axes. Exécutez-la après avoir codé votre routine. Ensuite, modifiez un paramètre, comparez deux expériences et justifiez votre lecture. Ce graphique isolé n’est pas encore la planète collective.

In [ ]:
try:
    plt.figure(figsize=(7,4))
    s,p=base();s.update(mantle_c_mol=100.,co2_mol=0.);p['volcanic_mol_yr']=10.
    h=planet_evolution(s,p,[step],15,1);x=[a['time_yr'] for a in h]
    plt.plot(x,[a['co2_mol'] for a in h],label='Atmosphère');plt.plot(x,[a['mantle_c_mol'] for a in h],label='Manteau')
    plt.xlabel('Temps (an)');plt.ylabel('Stock (mol)');plt.legend()
    plt.title("Groupe 3 — degazage")
    plt.grid(alpha=.25)
    plt.show()
except NotImplementedError as erreur:
    print("À compléter :", erreur)


## À VOUS — interprétation et raccordement

Prédiction avant la figure : …

Ce que montre la courbe : …

Explication par la formule : …

Limite de notre modèle : …

Voisin fournisseur et donnée de substitution à retirer : …

Sortie utilisée par un autre groupe : …

Contribution au graphe collectif et signe de la rétroaction : …

Livrer ce notebook complété, avec les résultats visibles et le nom des membres.

## Tester notre routine dans le projet commun

Le lien de classe Basthon charge aussi `classe_commune.py` depuis GitHub. Les autres groupes proviennent de la version commune affichée. Votre fonction locale remplace uniquement votre groupe, sans changer le dépôt. Si vous avez ouvert seulement le fichier .ipynb, charger aussi classe_commune.py comme module avec le bouton d’ouverture.

Une sauvegarde Basthon télécharge votre copie ; elle ne publie pas votre travail sur GitHub. Déposer le notebook et les résultats sur OneDrive. Après validation, le professeur met à jour la version commune ; les autres groupes la récupèrent au prochain chargement. Les références professeur restent identifiées tant qu’une routine élève n’a pas été validée.


In [ ]:
import json
try:
    from classe_commune import comparer_groupe, rapport_comparaison, VERSION_CLASSE
    print("Version commune :",VERSION_CLASSE)
    # Choix : chaude, gelee, deglaciation, societe, oxygenation.
    # Choisir un scénario qui sollicite votre mécanisme.
    resultat_classe=comparer_groupe(3,step,nom="deglaciation")
    page=rapport_comparaison(resultat_classe)
    from IPython.display import HTML, display
    from base64 import b64encode
    from html import escape
    display(HTML('<iframe title="Notre contribution" style="width:100%;height:900px;border:0" sandbox="allow-scripts allow-modals" srcdoc="'+escape(page,quote=True)+'"></iframe>'))
    for contenu,nom in [(page,"G03_rapport.html"),(json.dumps(resultat_classe,ensure_ascii=False),"G03_resultats.json")]:
        encoded=b64encode(contenu.encode()).decode()
        display(HTML('<p><a download="'+nom+'" href="data:application/octet-stream;base64,'+encoded+'">Télécharger '+nom+'</a></p>'))
except ImportError:
    print("Ouvrir le lien Basthon de classe, ou charger classe_commune.py comme module.")
except NotImplementedError as erreur:
    print("Compléter d’abord votre routine :",erreur)


## Notre bilan collectif

Quelle version commune avons-nous utilisée ? …

Quelles routines viennent de nos camarades et lesquelles restent des références ? …

Quel effet produit notre routine dans ce scénario ? …

Déposer le notebook, le rapport et les résultats dans le dossier du groupe.